# Prompt Preview

Ce notebook lit `data/raw/swissdox_input.parquet`, formate chaque article dans le template de prompt défini dans `src/prompts.py`, et produit un CSV avec :
- **`prompt_ready`** : le prompt complet à coller dans un chat LLM
- **`llm_answer`** : colonne vide à remplir manuellement avec la réponse du LLM

La colonne `text` brute est exclue pour alléger le fichier.

In [1]:
import sys
import pandas as pd
from pathlib import Path

# Ajouter src/ au path pour importer prompts.py
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from run1_prompts import build_user_prompt, SYSTEM_PROMPT

In [2]:
# Charger les données
df = pd.read_parquet('../data/raw/swissdox_input.parquet')
print(f"Articles chargés : {len(df):,}")
df.head(2)

Articles chargés : 8,787


,article_id,title,lead,text,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline
0,0001de76-bb46-1fd0-ff60-980d765e15a8,Mega-Angriff legt Schweizer Seiten lahm: Hacke...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,2025-01-21,ZWAO,20 minuten online,,None,WWE,Online medium,de,2666,
1,000658f2-9b7b-c9b4-9592-008f8f4d0b54,Vers la fin des vols en classe affaires pour l...,Privilèges Albert Rösti propose de réviser les...,Privilèges Albert Rösti propose de réviser les...,2025-09-30,HEU,24 heures,Actualités,None,PRD,Regional daily newspaper,fr,1612,


In [3]:
# Construire la colonne prompt_ready (system prompt + user prompt fusionnés)
df['prompt_ready'] = df.apply(
    lambda row: SYSTEM_PROMPT + "\n\n" + build_user_prompt(row, 'text'), axis=1
)

# Ajouter la colonne vide pour les réponses LLM
df['llm_answer'] = ''

print(f"Exemple de prompt_ready pour le premier article :")
print("-" * 60)
print(df['prompt_ready'].iloc[0])

Exemple de prompt_ready pour le premier article :
------------------------------------------------------------
You are a media analysis assistant specialised in Swiss public affairs.
Your task is to screen a newspaper article for two things and, when relevant, produce a concise summary.

--- STEP 1 — SWISS CONTEXT ---
Does the article have any connection to Switzerland?
Count as YES: any reference to a Swiss institution, official, law, city, company, currency, place, or affair — even minor.
Count as NO: zero connection to Switzerland.

--- STEP 2 — CRITICISM ---
(Evaluate only if STEP 1 = YES; otherwise answer NO.)

Is there any sentence or passage in the article where someone expresses a negative judgment about any person, institution, or entity?
A negative judgment = any statement that someone's conduct, decision, proposal, or inaction is wrong, harmful, inadequate, excessive, unjustified, or counterproductive.
A single qualifying sentence anywhere in the article is enough — the crit

In [4]:
# Supprimer la colonne text brute et exporter
cols_to_keep = [c for c in df.columns if c != 'text']
df_out = df[cols_to_keep]

out_path = Path('../data/raw/swissdox_prompts.csv')
df_out.to_csv(out_path, index=False)

print(f"Fichier exporté : {out_path}")
print(f"Colonnes : {df_out.columns.tolist()}")
print(f"Lignes : {len(df_out):,}")

Fichier exporté : ../data/raw/swissdox_prompts.csv
Colonnes : ['article_id', 'title', 'lead', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'prompt_ready', 'llm_answer']
Lignes : 8,787


In [5]:
# Aperçu du résultat final
df_out[['article_id', 'medium_name', 'pubtime', 'prompt_ready', 'llm_answer']].head(3)

,article_id,medium_name,pubtime,prompt_ready,llm_answer
0,0001de76-bb46-1fd0-ff60-980d765e15a8,20 minuten online,2025-01-21,You are a media analysis assistant specialised...,
1,000658f2-9b7b-c9b4-9592-008f8f4d0b54,24 heures,2025-09-30,You are a media analysis assistant specialised...,
2,0009a49c-b08a-7e36-0544-bdbeaec73b95,20 minuten online,2025-09-21,You are a media analysis assistant specialised...,
